In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from transformers import (
    RobertaTokenizer, 
    RobertaForMaskedLM, 
    LineByLineTextDataset, 
    DataCollatorForLanguageModeling, 
    TrainingArguments,
    Trainer
)

from datasets import load_dataset


/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c107WarningC1ENS_7variantIJNS0_11UserWarningENS0_18DeprecationWarningEEEERKNS_14SourceLocationENSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEEb'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/usr/local/lib/python3.10/dist-packages/torchvision/datapoints/__init__.py:14: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and yo

In [3]:
BLOCK_SIZE = 128 # Stride length when splitting long texts into 512-length segments
MAX_SEQ_LEN = 512 # maximum length of a sequence that BERT can operate on

In [4]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaForMaskedLM.from_pretrained('roberta-base')

In [5]:
# dataset = LineByLineTextDataset(
#     tokenizer=tokenizer,
#     file_path="temp/train_text.txt",
#     block_size=512,
# )

listing_txt_col = 'description'
resume_txt_col = 'text'

dataset = load_dataset('text', data_files={'train': './temp/train_text.txt'})
dataset = dataset['train'].train_test_split(test_size=0.2)
txt_col_name = 'text'


# dataset = load_dataset('eli5_category', split='train[:5000]')
# dataset = dataset.train_test_split(test_size=0.2)
# dataset = dataset.flatten()
# txt_col_name = 'answers.text'

In [6]:
# dataset['train'][3]
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 7200
    })
    test: Dataset({
        features: ['text'],
        num_rows: 1800
    })
})

In [63]:
def preprocess_eli5(x):
    '''preprocess for the ELI5 dataset'''
    return tokenizer([' '.join(sentence) for sentence in x[txt_col_name]])

def preprocess_hatespeech(x):
    '''preprocess for the Twitter hate speech dataset'''
    return tokenizer(x[txt_col_name])



tokenized_dataset = dataset.map(
    preprocess_hatespeech,
    batched=True,
    num_proc=4,
    remove_columns=dataset["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/7200 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1800 [00:00<?, ? examples/s]

In [66]:
# def group_texts(examples):
#     # Concatenate all texts.
#     concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
#     total_length = len(concatenated_examples[list(examples.keys())[0]])
#     # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
#     # customize this part to your needs.
#     if total_length >= BLOCK_SIZE:
#         total_length = (total_length // BLOCK_SIZE) * BLOCK_SIZE
#     # Split by chunks of BLOCK_SIZE.
#     result = {
#         k: [t[i : i + BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
#         for k, t in concatenated_examples.items()
#     }
#     print(len(result))
#     return result

def group_texts(token_data):
    '''
    Split texts that are two long (longer than 512 tokens) into overlapping segments with a sliding window of stride 128,
    and window size of 512. For example, a text with 1376 tokens is broken down into 8 segments that start at the 
    following indices:
        [0, 128, 256, 384, 512, 640, 768, 896]
    Then each segment has size:
        [512, 512, 512, 512, 512, 512, 512, 480]
    '''
    result = {k: [] for k in token_data.keys()}
    
    for key, data in token_data.items():
        for token_list in data:
            n_tokens = len(token_list)
            shifts = [t for t in range(0, n_tokens, BLOCK_SIZE) if t+MAX_SEQ_LEN-n_tokens < BLOCK_SIZE]
            if len(shifts) == 0:
                shifts = [0]

            for shift in shifts:
                result[key].append(token_list[shift : shift+MAX_SEQ_LEN])
    
    return result
    
    

In [67]:
lm_dataset = tokenized_dataset.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/7200 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1800 [00:00<?, ? examples/s]

In [68]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=True
)

In [69]:
training_args = TrainingArguments(
    output_dir="./temp/roberta-hatespeech",
    overwrite_output_dir=True,
    logging_strategy='epoch',
    eval_strategy='epoch',
    num_train_epochs=25,
#     learning_rate=2e-5,
    per_device_train_batch_size=32,
#     save_steps=500,
    save_strategy='epoch',
    save_total_limit=2,
    seed=1
)

In [70]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=lm_dataset['train'],
    eval_dataset=lm_dataset['test']
)

In [71]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.728600,2.348555
2,2.406700,2.303536
3,2.320100,2.292865
4,2.202900,2.244235
5,2.150400,2.234588
6,2.073800,2.178961
7,1.997400,2.216519
8,1.929500,2.244589
9,1.846200,2.155382
10,1.817800,2.193412


TrainOutput(global_step=5625, training_loss=1.805778271484375, metrics={'train_runtime': 305.7941, 'train_samples_per_second': 588.631, 'train_steps_per_second': 18.395, 'total_flos': 6889657174456320.0, 'train_loss': 1.805778271484375, 'epoch': 25.0})

In [73]:
trainer.save_model('./roberta-retrain-test')

In [91]:
from transformers import pipeline
fill_mask = pipeline(
    "fill-mask",
    model="./roberta-retrain-test",
    tokenizer="roberta-base"
)
fill_mask("What about Islamic <mask>?")

Device set to use cuda:0


[{'score': 0.3936074674129486,
  'token': 4498,
  'token_str': ' refugees',
  'sequence': 'What about Islamic refugees?'},
 {'score': 0.24860452115535736,
  'token': 2447,
  'token_str': ' immigration',
  'sequence': 'What about Islamic immigration?'},
 {'score': 0.09407435357570648,
  'token': 4870,
  'token_str': ' migrants',
  'sequence': 'What about Islamic migrants?'},
 {'score': 0.07262220233678818,
  'token': 4175,
  'token_str': ' immigrants',
  'sequence': 'What about Islamic immigrants?'},
 {'score': 0.02433311566710472,
  'token': 2040,
  'token_str': ' culture',
  'sequence': 'What about Islamic culture?'}]

In [ ]:
dataset[0]

In [ ]:
len(dataset[0]['input_ids'])

In [ ]:
# s = '@user nice new signage. Are you not concerned by Beatlemania -style hysterical crowds crongregating on you…'
# tokenizer.convert_ids_to_tokens(tokenizer.encode(s))

In [ ]:
!pip show accelerate